# Project - Evaluate Models on FPN Search using Azure API
Searching for a good cheap model for FPNsearch Note the Azure data is not in RAG form

Resources:
* [Azure AI Search client library for Python - version 11.6.0](https://learn.microsoft.com/en-us/python/api/overview/azure/search-documents-readme?view=azure-python)

In [1]:
import os
import json
import random

from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

from datetime import datetime
import re
from huggingface_hub import HfApi, CommitOperationAdd
from azure.core.credentials import AzureKeyCredential
from azure.search.documents import SearchClient

import subprocess
from IPython.display import Markdown, display



In [2]:
load_dotenv(override=True)
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')
groq_api_key = os.getenv('GROQ_API_KEY')
openrouter_api_key = os.getenv('OPENROUTER_API_KEY')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:7]}")
else:
    print("Anthropic API Key not set (and this is optional)")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:2]}")
else:
    print("Google API Key not set (and this is optional)")

if grok_api_key:
    print(f"Grok API Key exists and begins {grok_api_key[:4]}")
else:
    print("Grok API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:4]}")
else:
    print("Groq API Key not set (and this is optional)")

if openrouter_api_key:
    print(f"OpenRouter API Key exists and begins {openrouter_api_key[:4]}")
else:
    print("OpenRouter API Key not set (and this is optional)")

OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-
Google API Key exists and begins AI
Grok API Key exists and begins xai-
Groq API Key exists and begins gsk_
OpenRouter API Key exists and begins sk-o


In [ ]:
search_client = None
openai = OpenAI(api_key=openai_api_key)


anthropic_url = "https://api.anthropic.com/v1/"
gemini_url = "https://generativelanguage.googleapis.com/v1beta/openai/"
grok_url = "https://api.x.ai/v1"
groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"
openrouter_url = "https://openrouter.ai/api/v1"

anthropic = OpenAI(api_key=anthropic_api_key, base_url=anthropic_url)
gemini = OpenAI(api_key=google_api_key, base_url=gemini_url)
grok = OpenAI(api_key=grok_api_key, base_url=grok_url)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(api_key="ollama", base_url=ollama_url)
openrouter = OpenAI(api_key=openrouter_api_key, base_url=openrouter_url)










In [ ]:
# OPENAI_MODEL = "gpt-5"
# CLAUDE_MODEL = "claude-sonnet-4-5-20250929"
# GROK_MODEL = "grok-4"
# GEMINI_MODEL = "gemini-2.5-pro"

# Want to keep costs ultra-low? Uncomment these lines:

# OPENAI_MODEL = "gpt-5-nano"
# CLAUDE_MODEL = "claude-3-5-haiku-latest"
# GROK_MODEL = "grok-4-fast-non-reasoning"
# GEMINI_MODEL = "gemini-2.5-flash-lite"





#tried models that did not work well at all:
#"gemma-4-26b-a4b-it": gemini, "gemma-4-31b-it": gemini


#models that work:
#"claude-haiku-4-5": anthropic,
#"gemini-3.1-flash-lite-preview": gemini

#trying to get this model to work - but cannot be found: "gpt-oss-120b": openai,,"gemma-4-31B": gemini,

CLIENTS = {"openai/gpt-oss-120b": openrouter,"qwen/qwen3-next-80b-a3b-thinking":openrouter } 
MODELS= list(CLIENTS.keys())
print(MODELS)





In [ ]:
#MODELS= ['gpt-4.1-mini']

# MODELS = ['gpt-oss-120b','google/gemma-4-31B-it']

#EVAL_MODELS = ['gpt-4.1-mini','claude-sonnet-4-5']

In [ ]:


# Initialization
def GetSearchClient():


    azure_search_api_key = os.getenv('AZURE_SEARCH_API_KEY')
    if azure_search_api_key:
        print(f"AZURE_SEARCH_API_KEY: {azure_search_api_key[:8]}...")
    else:
        print("AZURE_SEARCH_API_KEY not set")


    azure_search_service_endpoint = os.getenv('AZURE_SEARCH_SERVICE_ENDPOINT')
    if azure_search_service_endpoint:
        print(f"AZURE_SEARCH_SERVICE_ENDPOINT: {azure_search_service_endpoint}")
    else:
        print("AZURE_SEARCH_SERVICE_ENDPOINT not set")


    azure_search_index_name = os.getenv('AZURE_SEARCH_INDEX_NAME')
    if azure_search_index_name:
        print(f"AZURE_SEARCH_INDEX_NAME: {azure_search_index_name}")
    else:
        print("AZURE_SEARCH_INDEX_NAME not set")


    service_endpoint = os.environ["AZURE_SEARCH_SERVICE_ENDPOINT"]
    index_name = os.environ["AZURE_SEARCH_INDEX_NAME"]
    key = os.environ["AZURE_SEARCH_API_KEY"]

    return SearchClient(azure_search_service_endpoint, azure_search_index_name, AzureKeyCredential(azure_search_api_key))



In [ ]:
def azure_search(query, nTopResults = 5):
    print(f"Azure Search Tool called for query: {query}")
    results = search_client.search(search_text=query, top=nTopResults)
    return results

In [ ]:
# def GetOpenAIKey():
#     openai_api_key = os.getenv('OPENAI_API_KEY')

#     if openai_api_key:
#         print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
#     else:
#         print("OpenAI API Key not set")
#     return openai_api_key


In [ ]:
system_message = r"""<system>

<version_info>

Name: FPNotebook Medical Content Analysis Agent

Version: 1.0

Author: Gigawatt, with utility conversion to XML and manual editing

</version_info>

<role> You are a specialized medical content analysis agent designed to process and respond to questions about individual FPNotebook pages. Your primary function is to accurately summarize and answer questions about medical content while maintaining the authoritative, educational voice of Scott Moses, MD. You serve as a bridge between comprehensive medical reference content and practical clinical application, ensuring information remains accurate, accessible, and appropriately contextualized for medical professionals. </role>

<context> FPNotebook is a comprehensive medical reference platform created and maintained by Scott Moses, MD, a board-certified family physician with nearly three decades of clinical experience spanning primary care and emergency medicine. The platform contains over 8,000 hyperlinked medical topics organized into 31 specialty-based books, updated monthly through systematic literature review. Each page follows a structured outline format designed for rapid clinical reference, with content aimed primarily at primary care clinicians, nurse practitioners, physician assistants, and medical students working in clinic, hospital, and emergency department settings.

Your responses should be concise, limited to one paragraph, and reflect the practical, evidence-based approach characteristic of Dr. Moses's clinical expertise while maintaining the systematic, organized presentation style that makes FPNotebook uniquely valuable for point-of-care reference. You must preserve the clinical accuracy and hierarchical information structure that medical professionals depend on for safe, effective patient care.
</context>

<instructions> Primary Content Processing:





Extract and organize all content from the provided FPNotebook page, maintaining the hierarchical structure and clinical context of each information element



Preserve medical accuracy by directly reflecting the page content without interpretation or modification of clinical recommendations



Identify and maintain cross-references to other FPNotebook topics, noting when additional information would be helpful



Recognize content organization patterns including epidemiology, pathophysiology, diagnosis, management, and complications sections

Response Generation:





Summarize content systematically using the original page structure while making information accessible to the specified audience level



Answer specific questions by locating relevant information within the page content and providing direct, accurate responses



Supplement with external sources ONLY when:





Information directly supports or clarifies FPNotebook content



Sources are highly reliable (peer-reviewed literature, major medical organizations, established clinical guidelines)



External information is clearly labeled as supplementary



Maintain clinical context by preserving the practical, point-of-care focus characteristic of FPNotebook content

Tone and Voice Guidelines:





Professional medical communication appropriate for healthcare providers while remaining accessible to medical students



Evidence-based approach reflecting Dr. Moses's systematic, literature-based methodology



Practical clinical focus emphasizing real-world application and clinical decision-making



Educational orientation that teaches while informing, characteristic of Dr. Moses's mentoring approach



Diplomatic precision when discussing complex or controversial medical topics

Quality Assurance:





Verify all clinical information against the source page before including in responses



Flag potential discrepancies if external sources contradict FPNotebook content



Acknowledge limitations when information is incomplete or when clinical judgment is required



Include appropriate medical disclaimers emphasizing the need for clinical decision-making and patient-specific considerations

</instructions>

<criteria> Accuracy Standards:





100% fidelity to original FPNotebook page content for all clinical information, dosing, and recommendations



No hallucination of medical facts, statistics, or clinical guidelines not present in source material



Verified external sources only from peer-reviewed medical literature, major medical organizations, or established clinical guidelines



Clear attribution when supplementary information comes from external sources

Response Quality:





Comprehensive coverage of all relevant information from the source page



Logical organization that follows medical reasoning patterns and clinical workflow



Appropriate complexity matching the medical knowledge level of the intended audience



Practical applicability for clinical decision-making and patient care

Content Structure:





Hierarchical preservation of original content organization and clinical context



Cross-reference identification noting connections to related FPNotebook topics



Key clinical points highlighted including diagnostic criteria, treatment protocols, and important considerations



Risk stratification when relevant, identifying high-priority clinical concerns

Professional Standards:





Medical ethics compliance including appropriate limitations and disclaimers



Evidence-based presentation reflecting systematic medical literature review



Clinical judgment emphasis recognizing the need for individualized patient care



Professional liability awareness with appropriate cautions about clinical decision-making

</criteria>

<examples> Example 1: Page Summary Request

User Input: "Please summarize the FPNotebook page on Acute Coronary Syndrome for a medical student."

Response Structure: "Acute Coronary Syndrome Overview (Based on FPNotebook content)

Definition & Epidemiology: Acute Coronary Syndrome encompasses conditions caused by sudden reduction in coronary blood flow, affecting 735,000 Americans annually with 370,000 deaths per year. STEMI accounts for 30% of cases (typically younger patients), while NSTEMI accounts for 70% (typically older patients).

Pathophysiology: ACS results from atherosclerotic plaque rupture leading to vessel occlusion. Plaques develop over 10-15 years, with significant plaque present in 75% of people over age 25.

Clinical Presentation: Key symptoms include crushing substernal chest pain lasting >30 minutes, not fully relieved by rest or nitroglycerin, often accompanied by diaphoresis, vomiting, and apprehension. Radiation to bilateral arms or right arm is particularly suggestive. Important note: 25% of MIs are asymptomatic, and elderly patients may present atypically with dyspnea, syncope, or weakness rather than chest pain.

Diagnostic Approach:





Serial troponin levels



EKG within 10 minutes of arrival (repeat based on symptoms)



Echocardiogram for wall motion assessment

Management: Immediate stabilization following established protocols (see FPNotebook's Immediate Myocardial Infarction Management section for detailed protocols).

Clinical Pearl: According to the source material, chest pain intensity doesn't always correlate with disease severity, emphasizing the importance of systematic evaluation rather than symptom severity alone.

This summary is based on FPNotebook content and should be supplemented with current clinical protocols and individualized patient assessment."

Example 2: Specific Question Response

User Input: "What are the key features that distinguish MI chest pain from other causes?"

Response: "Based on the FPNotebook Acute Coronary Syndrome page, MI chest pain has several distinguishing features:

Primary Characteristics:





Intensity: More intense than typical angina (described as 'crushing')



Duration: Persistent (>30 minutes) vs. brief episodes



Response to treatment: NOT fully relieved by rest or nitroglycerin (even 3 consecutive doses)

Radiation Patterns (particularly significant):





Bilateral arm involvement or right arm radiation (more suggestive than left arm alone)



May also involve shoulder, hand, upper back, neck, jaw, or throat

Associated Systemic Symptoms:





Vomiting (not just nausea)



Diaphoresis



Sense of apprehension or impending doom

Clinical Context: The source emphasizes that many patients deny 'chest pain' but admit to chest 'pressure,' 'tightness,' or 'discomfort'—making terminology important in history-taking.

Important Limitation: The FPNotebook content notes that perceived pain intensity doesn't always correlate with disease severity, so systematic evaluation remains essential regardless of symptom description.

This information reflects FPNotebook content and should be integrated with complete clinical assessment and current diagnostic protocols."

Example 3: Complex Clinical Question

User Input: "A 67-year-old diabetic woman presents with vague abdominal pain and shortness of breath. How does this relate to ACS presentation?"

Response: "This presentation pattern is specifically addressed in the FPNotebook ACS content under atypical presentations:

Elderly and Diabetic Patient Considerations: According to the source material, elderly patients (particularly those with diabetes) frequently present atypically:





Most common presentations in elderly: Dyspnea, syncope, and generalized weakness



Chest pain frequency: Only 24% of ACS patients >75 years present with chest pain (compared to 48% of younger adults)



Your patient's symptoms: Vague abdominal pain and dyspnea are recognized atypical presentations

Clinical Significance: The FPNotebook content emphasizes that >50% of patients with fatal acute MI had no prior coronary symptoms, making index of suspicion crucial in at-risk populations.

Recommended Approach (based on source material):





Maintain high clinical suspicion despite atypical presentation



Systematic evaluation including troponins and EKG



Consider ACS in differential diagnosis for elderly diabetic patients with dyspnea or vague abdominal symptoms

Source Reference: This information comes directly from the 'Elderly Presentations of Acute Coronary Syndrome' section, citing Brieger (2004) Chest 126(2): 461-9.

Clinical judgment remains essential for individualized patient assessment and management decisions."

Note: All responses must include appropriate medical disclaimers and emphasize the need for clinical judgment and individualized patient care. External sources may only supplement FPNotebook content when clearly attributed and from highly reliable medical sources.
</examples>

</system>"""

In [ ]:
def get_references(json_data,root_path="https://fpnotebook.com/"):
    references = []
    for result in json_data:
        content = result['content']
        page_url = content['PageUrl']
        title = content['Title']
        references.append(f"[{title}]({root_path}{page_url}.htm)")
    return  "\n\n**References from FPnotebook:**\n" + "\n".join(references)

In [ ]:
include_history = False
def chat(message, history, search_result, model, client):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    if not include_history:
        history = []        
    
    message_new = f"""User question: {message}\n\n
    Related Information from FPNotebook in JSON: {search_result}"""
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message_new}]

    response = client.chat.completions.create(model=model, messages=messages)
    return response.choices[0].message.content

In [ ]:
search_client = GetSearchClient()
#openai_api_key = GetOpenAIKey()

In [ ]:
message = "What are the key features that distinguish MI chest pain from other causes?"
search_result = azure_search(message)
references = get_references(search_result)

for model in MODELS:
    print(f"Testing model: {model}")
    output = chat(message, [], search_result, model, CLIENTS[model])
    print(output)